# 4 · La campaña "El sur tiene espacio"
**Torre del Caribe** · Mercadotecnia digital (criterio 7) y minería de texto (criterio 3) · Autor: **Brandon Uriel García
Sánchez** · UNRC, LCDN 5° semestre 2026-2

Qué molesta y qué enamora al turista (85,987 reseñas), las dos viajeras ideales, la marca, los anuncios, los medios, el
calendario y los indicadores. Cada rasgo dice si es un dato, un cálculo, un supuesto o un hueco.

| | |
|---|---|
| **Cómo correrlo** | Python 3.11 con `pip install -r requirements.txt` (en la carpeta de la entrega). Para Spark (notebooks 1 y 3), además Java 17: `winget install Microsoft.OpenJDK.17`. Abrir en Jupyter o VS Code y elegir *Ejecutar todo*. |
| **Datos** | Los lee de `datos/` (tablas limpias y resultados). Ya viene ejecutado: se puede leer sin correrlo. |
| **Código** | Va dentro del notebook: cada celda que empieza con `%%modulo` es un archivo del proyecto, completo. |

In [1]:
%matplotlib inline
# Preparación: se corre una vez, al inicio. Encuentra la carpeta de la entrega y define %%modulo, la instrucción que
# convierte una celda de código en un módulo del paquete `torre` (así el notebook no necesita archivos .py).
import os, sys, types, warnings
from pathlib import Path
from IPython.core.magic import register_cell_magic
warnings.filterwarnings("ignore")

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos" / "silver").is_dir())
FIGURAS = RAIZ / "2_Notebooks" / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)


def _paquete(nombre):
    if nombre not in sys.modules:
        m = types.ModuleType(nombre); m.__path__ = []
        sys.modules[nombre] = m
        if "." in nombre:
            padre, hijo = nombre.rsplit(".", 1)
            setattr(_paquete(padre), hijo, m)
    return sys.modules[nombre]


@register_cell_magic
def modulo(linea, celda):
    """%%modulo torre.radar.indice → ejecuta la celda como el módulo torre.radar.indice."""
    nombre = linea.strip()
    padre, hijo = nombre.rsplit(".", 1)
    m = types.ModuleType(nombre)
    # Ruta simbólica: el código calcula la raíz del proyecto con Path(__file__).parents[3], y aquí da la carpeta de entrega.
    m.__file__ = str(RAIZ / "codigo" / Path(*nombre.split(".")).with_suffix(".py"))
    sys.modules[nombre] = m
    setattr(_paquete(padre), hijo, m)
    try:  # para que Spark pueda mandar a sus trabajadores las funciones definidas aquí
        from pyspark import cloudpickle
        cloudpickle.register_pickle_by_value(m)
    except Exception:
        pass
    exec(compile(celda, m.__file__, "exec"), m.__dict__)


import pandas as pd
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)
print("Carpeta de la entrega:", RAIZ)

Carpeta de la entrega: C:\Users\Uriel\Desktop\PP 5to semestre CASCO SANTO TOMAS\Entregable


### El código de la campaña
Cada celda de código de abajo es un archivo del paquete `torre`, completo y sin cambios (con su encabezado: qué hace, por qué así, de qué datos sale y a qué decisión alimenta). Al correrla, la celda queda registrada como ese módulo y las celdas siguientes lo usan.

- **`torre.base.entorno`** — Prepara el entorno para que PySpark funcione en Windows y crea la sesión de Spark.
- **`torre.campana.texto`** — Minería de las 85,987 reseñas de Quintana Roo de Rest-Mex 2025 (Tulum, Isla Mujeres y Bacalar): 1. Temas por aspecto con un léxico transparente (multitudes, precio, limpieza, ruido, calma, naturaleza, cultura, comida, servicio): cuántas reseñas tocan cada tema y qué tan seguido es una reseña MALA (1–2 estrellas) cuando lo toca (riesgo relativo). 2. Reglas de asociación (Apriori, mlxtend): qué combinaciones de temas llevan a una reseña mala o a una de 5 estrellas (soporte, confianza y lift). 3. El lenguaje real del turista: palabras que más distinguen a las reseñas de 5 estrellas de las de 1–2 (log-odds con prior de Dirichlet informativo, Monroe et al. 2008). Con ellas se escriben los mensajes de la campaña.
- **`torre.campana.marca`** — Arma la campaña "El sur tiene espacio" con los 10 elementos del Problema Prototípico que le tocan: buyer persona (2), branding (identidad, personalidad, propuesta de valor, posicionamiento, elementos visuales), mensajes por persona y canal, medios justificados, piezas, implementación (calendario) e indicadores (KPI). Cada atributo de persona lleva su etiqueta: DATO (medido), DERIVADO (calculado de datos) o SUPUESTO (no hay dato; se declara). Cada mensaje lleva el dato que lo respalda.

In [2]:
%%modulo torre.base.entorno
# Autor: Brandon Uriel García Sánchez
# Módulo: Base de datos
# Qué hace:          Prepara el entorno para que PySpark funcione en Windows y crea la sesión de Spark.
# Por qué así:       En esta máquina el Java por defecto es 1.8 (verificado el 26-sep-2026) y PySpark necesita
#                    Java 17. En lugar de cambiar el Java de todo Windows, este módulo apunta JAVA_HOME al JDK 17
#                    solo para el proceso del proyecto. HADOOP_HOME apunta a herramientas/hadoop (winutils.exe y
#                    hadoop.dll 3.3.6), sin los cuales Spark no puede escribir Parquet en Windows.
#                    Alternativa descartada: correr Spark en Docker (funciona, pero es más pesado de explicar y de
#                    arrancar el día del coloquio).
# Datos de entrada:  Ninguno (solo configura el entorno).
# Alimenta a:        Todas las tuberías de Bronze → Silver → Gold (criterio 2 de la rúbrica).

import glob
import os
import sys
from pathlib import Path

# Raíz del proyecto: este archivo vive en backend/torre/base/, así que subimos tres niveles.
RAIZ = Path(__file__).resolve().parents[3]
HADOOP_HOME = RAIZ / "herramientas" / "hadoop"


def buscar_jdk17() -> Path:
    """Busca un JDK 17 instalado en las rutas estándar de Windows.

    Devuelve la carpeta del JDK. Si no hay uno, lanza un error que dice exactamente
    cómo instalarlo; no se sigue con otro Java porque PySpark fallaría más adelante.
    """
    candidatos = []
    for patron in (r"C:\Program Files\Microsoft\jdk-17*",
                   r"C:\Program Files\Eclipse Adoptium\jdk-17*",
                   r"C:\Program Files\Java\jdk-17*"):
        candidatos.extend(glob.glob(patron))
    if not candidatos:
        raise RuntimeError(
            "No encontré un JDK 17. Instálalo con:\n"
            "    winget install Microsoft.OpenJDK.17\n"
            "y vuelve a correr la prueba."
        )
    # Si hay varios, se usa el de versión más alta. Como texto, "jdk-17.0.9" quedaría después de
    # "jdk-17.0.20", así que se ordena por los números de la versión y no alfabéticamente.
    def version(ruta: str) -> tuple:
        numeros = "".join(c if c.isdigit() else " " for c in Path(ruta).name).split()
        return tuple(int(n) for n in numeros)
    return Path(max(candidatos, key=version))


def configurar_entorno() -> dict:
    """Fija JAVA_HOME, HADOOP_HOME y PATH solo para este proceso. Devuelve lo que configuró."""
    jdk = buscar_jdk17()
    os.environ["JAVA_HOME"] = str(jdk)
    os.environ["HADOOP_HOME"] = str(HADOOP_HOME)
    # Se antepone al PATH para que gane sobre el Java 8 del sistema.
    os.environ["PATH"] = os.pathsep.join([str(jdk / "bin"), str(HADOOP_HOME / "bin"), os.environ["PATH"]])
    # Spark arranca "trabajadores" de Python cuando recibe datos creados en Python. En Windows, si no se le dice qué
    # Python usar, se queda esperando y falla con "Accept timed out" (visto el 28-sep-2026 al construir la ocupación
    # de DataTur). Se le indica el mismo Python del proyecto (.venv).
    python = _ruta_corta(sys.executable)
    os.environ["PYSPARK_PYTHON"] = python
    os.environ["PYSPARK_DRIVER_PYTHON"] = python
    return {"JAVA_HOME": str(jdk), "HADOOP_HOME": str(HADOOP_HOME), "PYSPARK_PYTHON": python}


def _ruta_corta(ruta: str) -> str:
    """Devuelve la ruta "corta" de Windows, sin espacios (por ejemplo, la carpeta "PP 5to semestre..." pasa a "PP5TOS~1").

    La carpeta del proyecto tiene espacios ("PP 5to semestre ...") y los scripts de Spark la cortan en "5to"
    ("'5to' is not recognized", visto el 28-sep-2026). La ruta corta evita el problema. Fuera de Windows no cambia nada.
    """
    if os.name != "nt":
        return ruta
    import ctypes

    buffer = ctypes.create_unicode_buffer(1024)
    n = ctypes.windll.kernel32.GetShortPathNameW(ruta, buffer, len(buffer))
    return buffer.value if 0 < n < len(buffer) else ruta


def crear_spark(nombre_app: str = "torre-del-caribe"):
    """Crea una sesión de Spark local que usa todos los núcleos de la máquina.

    local[*]    = modo local con todos los núcleos (12 en esta máquina).
    driver 4g   = memoria para el proceso principal (la máquina tiene 16 GB).
    timeZone    = UTC, para que las fechas no se muevan entre máquinas.
    progreso    = apagado, para que los resúmenes se lean limpios.
    """
    configurar_entorno()
    from pyspark.sql import SparkSession  # se importa después de fijar JAVA_HOME

    return (
        SparkSession.builder.appName(nombre_app)
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .config("spark.sql.session.timeZone", "UTC")
        # Sin barra de progreso en la terminal: tapaba los resúmenes impresos por los procesos.
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )


In [3]:
%%modulo torre.campana.texto
# Autor: Brandon Uriel García Sánchez
# Módulo: Campaña (minería de texto, Fase 8)
# Qué hace:          Minería de las 85,987 reseñas de Quintana Roo de Rest-Mex 2025 (Tulum, Isla Mujeres y Bacalar):
#                    1. Temas por aspecto con un léxico transparente (multitudes, precio, limpieza, ruido, calma,
#                       naturaleza, cultura, comida, servicio): cuántas reseñas tocan cada tema y qué tan seguido es una
#                       reseña MALA (1–2 estrellas) cuando lo toca (riesgo relativo).
#                    2. Reglas de asociación (Apriori, mlxtend): qué combinaciones de temas llevan a una reseña mala o a
#                       una de 5 estrellas (soporte, confianza y lift).
#                    3. El lenguaje real del turista: palabras que más distinguen a las reseñas de 5 estrellas de las de
#                       1–2 (log-odds con prior de Dirichlet informativo, Monroe et al. 2008). Con ellas se escriben los
#                       mensajes de la campaña.
# Por qué así:       - Léxico y no un modelo de temas (LDA/NMF): se puede leer, explicar y defender palabra por palabra, como
#                      el NLP de lugares.py. Un tema de LDA hay que adivinarlo; un tema del léxico se define.
#                    - Ninguna reseña es de los 5 lugares (Rest-Mex no los cubre, nota 18): el texto dice qué MOLESTA en
#                      los destinos llenos y qué valora el turista; eso sostiene la promesa del sur, no una opinión del sur.
#                    - Muchas reseñas son traducciones al español (se nota en el texto): se usan raíces cortas para tolerar
#                      variantes y se declara.
# Datos de entrada:  datos/silver/restmex (D5).
# Alimenta a:        Buyer persona, propuesta de valor y mensajes de la campaña (Fase 8).

import re
from collections import Counter

import numpy as np
import pandas as pd

from torre.base.entorno import RAIZ

SILVER = RAIZ / "datos" / "silver"
GOLD = RAIZ / "datos" / "gold"

# Raíces al inicio de palabra (\b). Cada tema se define aquí, a la vista.
ASPECTOS = {
    "multitudes": ["gente", "lleno", "llena", "multitud", "abarrot", "saturad", "concurrid", "masific", "aglomer",
                   "fila", "filas", "esperar", "espera ", "demasiados turistas", "muchos turistas"],
    "precio": ["caro", "cara", "caros", "caras", "costos", "precio", "cobr", "tarifa", "propina", "barat", "dinero"],
    "limpieza": ["sucio", "sucia", "sucied", "basura", "limpi", "sargazo", "olor", "apest", "mugr"],
    "ruido": ["ruido", "ruidos", "escándalo", "música alta", "fiesta"],
    "calma": ["tranquil", "paz", "relaj", "silencio", "calma", "sereno", "serena"],
    "naturaleza": ["naturalez", "selva", "laguna", "cenote", "manglar", "aves", "pájaro", "agua cristalina", "colores"],
    "cultura": ["ruina", "pirámide", "piramide", "maya", "historia", "arqueol", "cultura", "templo"],
    "comida": ["comida", "platillo", "delicios", "sabor", "mariscos", "tacos", "ceviche", "pescado"],
    "servicio": ["personal", "servicio", "atención", "amable", "mesero", "anfitri", "recepción"],
}
PALABRAS_VACIAS = set("""a al algo algunas algunos ante antes aquí así aun aunque bien cada casi como con contra cual cuales
cuando de del desde donde dos e el ella ellas ellos en entre era eran es esa esas ese eso esos esta estaba estaban estamos
estan están estar estas este esto estos estuvo fue fueron fuimos ha había habían han hasta hay hemos hizo hubo la las le les
lo los mas más me mi mis mucho muchos muy nada ni no nos nosotros o os otra otras otro otros para pero poco por porque que
qué quien se sea sean ser si sí sido sin sobre solo sólo son su sus también tan tanto te tenía tenían tiene tienen todo
todos tu tus tuvimos tuvo un una unas uno unos y ya él ese esa hace hacer puede pueden día días vez veces lugar lugares
cosa cosas así cuál dentro fuera luego entonces mientras ser estar haber tener ir ver dar decir mismo misma después aquí
allí allá ahí cómo dónde etc gran grande buen buena bueno buenos buenas mejor mal mala malo""".split())


def leer() -> pd.DataFrame:
    r = pd.read_parquet(SILVER / "restmex", filters=[("estado", "=", "Quintana Roo")],
                        columns=["titulo", "resena", "calificacion", "pueblo", "tipo"])
    r["texto"] = (r.titulo.fillna("") + ". " + r.resena.fillna("")).str.lower()
    return r


def marcar_aspectos(r: pd.DataFrame) -> pd.DataFrame:
    for a, raices in ASPECTOS.items():
        patron = r"\b(?:" + "|".join(re.escape(x.strip()) for x in raices) + ")"
        r[a] = r.texto.str.contains(patron, regex=True)
    r["mala"] = r.calificacion <= 2
    r["cinco"] = r.calificacion == 5
    return r


def aspectos(r: pd.DataFrame) -> pd.DataFrame:
    """Por pueblo y aspecto: % de reseñas que lo tocan, % malas entre ellas y riesgo relativo (vs. todas las del pueblo)."""
    filas = []
    for pueblo, g in list(r.groupby("pueblo")) + [("Quintana Roo (3 pueblos)", r)]:
        base = g.mala.mean()
        for a in ASPECTOS:
            t = g[g[a]]
            filas.append({"pueblo": pueblo, "aspecto": a, "resenas": len(t), "pct_menciona": 100 * len(t) / len(g),
                          "pct_malas": 100 * t.mala.mean() if len(t) else np.nan, "pct_malas_pueblo": 100 * base,
                          "riesgo_relativo": t.mala.mean() / base if len(t) and base else np.nan})
    return pd.DataFrame(filas)


def reglas_asociacion(r: pd.DataFrame, soporte: float = 0.001) -> pd.DataFrame:
    """Apriori sobre transacciones {temas mencionados} ∪ {reseña mala | reseña de 5}. Reglas que terminan en la
    calificación. lift = confianza / soporte(consecuente)."""
    from mlxtend.frequent_patterns import apriori, association_rules
    t = r[list(ASPECTOS)].copy()
    t["reseña mala (1–2)"] = r.mala
    t["reseña de 5"] = r.cinco
    frec = apriori(t.astype(bool), min_support=soporte, use_colnames=True, max_len=3)
    reg = association_rules(frec, metric="lift", min_threshold=1.0)
    fin = reg.consequents.apply(lambda c: len(c) == 1 and next(iter(c)) in {"reseña mala (1–2)", "reseña de 5"})
    ante = reg.antecedents.apply(lambda a: not (a & {"reseña mala (1–2)", "reseña de 5"}))
    reg = reg[fin & ante].copy()
    reg["si"] = reg.antecedents.apply(lambda a: " + ".join(sorted(a)))
    reg["entonces"] = reg.consequents.apply(lambda c: next(iter(c)))
    return reg[["si", "entonces", "support", "confidence", "lift"]].rename(
        columns={"support": "soporte", "confidence": "confianza"}).sort_values(["entonces", "lift"], ascending=[True, False])


def tokens(texto: str) -> list[str]:
    return [w for w in re.findall(r"[a-záéíóúñü]{3,}", texto) if w not in PALABRAS_VACIAS]


def palabras_que_distinguen(r: pd.DataFrame, n: int = 20) -> pd.DataFrame:
    """Log-odds con prior de Dirichlet informativo (Monroe, Colaresi y Quinn, 2008):
    δ_w = ln[(y_w^A + α_w)/(n_A + α_0 − y_w^A − α_w)] − ln[(y_w^B + α_w)/(n_B + α_0 − y_w^B − α_w)],
    z_w = δ_w / √(1/(y_w^A+α_w) + 1/(y_w^B+α_w)), con α_w = 0.01·(conteo de w en todo el corpus) (prior informativo)."""
    a = Counter(w for t in r[r.cinco].texto for w in tokens(t))
    b = Counter(w for t in r[r.mala].texto for w in tokens(t))
    todo = a + b
    alfa = {w: 0.01 * c for w, c in todo.items()}
    a0 = sum(alfa.values())
    na, nb = sum(a.values()), sum(b.values())
    filas = []
    for w, c in todo.items():
        if c < 50:
            continue
        ya, yb, al = a[w], b[w], alfa[w]
        d = np.log((ya + al) / (na + a0 - ya - al)) - np.log((yb + al) / (nb + a0 - yb - al))
        z = d / np.sqrt(1 / (ya + al) + 1 / (yb + al))
        filas.append({"palabra": w, "en_cinco": ya, "en_malas": yb, "delta": d, "z": z})
    f = pd.DataFrame(filas)
    cinco = f.nlargest(n, "z").assign(lado="reseñas de 5 estrellas")
    malas = f.nsmallest(n, "z").assign(lado="reseñas de 1–2 estrellas")
    return pd.concat([cinco, malas], ignore_index=True)


def construir() -> dict:
    r = marcar_aspectos(leer())
    asp, reg, pal = aspectos(r), reglas_asociacion(r), palabras_que_distinguen(r)
    asp.to_parquet(GOLD / "campana_aspectos.parquet", index=False)
    reg.to_parquet(GOLD / "campana_reglas.parquet", index=False)
    pal.to_parquet(GOLD / "campana_palabras.parquet", index=False)
    print(f"Reseñas: {len(r):,} · malas (1–2): {r.mala.mean():.1%} · de 5: {r.cinco.mean():.1%}")
    q = asp[asp.pueblo == "Quintana Roo (3 pueblos)"].sort_values("riesgo_relativo", ascending=False)
    print(q[["aspecto", "pct_menciona", "pct_malas", "riesgo_relativo"]].round(2).to_string(index=False))
    print(asp.pivot(index="aspecto", columns="pueblo", values="riesgo_relativo").round(2).to_string())
    print(reg[reg.entonces.str.contains("mala")].head(12).round(4).to_string(index=False))
    print(reg[reg.entonces == "reseña de 5"].head(8).round(3).to_string(index=False))
    print(pal.groupby("lado").palabra.apply(lambda p: ", ".join(p)).to_string())
    return {"r": r, "aspectos": asp, "reglas": reg, "palabras": pal}


In [4]:
%%modulo torre.campana.marca
# Autor: Brandon Uriel García Sánchez
# Módulo: Campaña (mercadotecnia, Fase 8)
# Qué hace:          Arma la campaña "El sur tiene espacio" con los 10 elementos del Problema Prototípico que le tocan:
#                    buyer persona (2), branding (identidad, personalidad, propuesta de valor, posicionamiento, elementos
#                    visuales), mensajes por persona y canal, medios justificados, piezas, implementación (calendario) e
#                    indicadores (KPI). Cada atributo de persona lleva su etiqueta: DATO (medido), DERIVADO (calculado de
#                    datos) o SUPUESTO (no hay dato; se declara). Cada mensaje lleva el dato que lo respalda.
# Por qué así:       Decisiones del equipo del 02-oct-2026 (docs/decisiones/21-campana.md): dos personas (nacional que
#                    vuelve; extranjera que ya está en Cancún), nombre "El sur tiene espacio", tono que combina lo
#                    cercano y tranquilo, el orgullo cultural y la aventura ("es un orgullo ser mexicano").
#                    - Un mensaje no puede prometer lo que el dato no sostiene: "sin multitudes" se apoya en 15.5 veces
#                      menos visitantes que Tulum (INAH 2025); no se dan horas de viaje ni precios (no hay dato).
#                    - El lenguaje sale de las reseñas de 5 estrellas (texto.py) y evita lo que hunde una reseña (ruido,
#                      suciedad, precio, multitudes).
#                    - Los largos de los anuncios se revisan con los límites de cada plataforma (Google: título ≤ 30,
#                      descripción ≤ 90; Facebook/Instagram: título ≤ 40, texto ≤ 125 recomendado).
# Datos de entrada:  datos/silver/inah, nacionalidad; datos/gold/campana_*, presupuesto_*, pronostico_*, envivo_*;
#                    ENDUTIH 2025 (D11, cifras transcritas del PDF oficial con su página).
# Alimenta a:        La sección "La campaña" de la página, el capítulo 12 del documento ejecutivo y el coloquio.

import json

import pandas as pd

from torre.base.entorno import RAIZ

SILVER = RAIZ / "datos" / "silver"
GOLD = RAIZ / "datos" / "gold"
SALIDA = GOLD / "campana.json"

# ENDUTIH 2025 (INEGI, reporte de resultados del 16-jun-2026): cifras leídas del PDF de Bronze, con su página.
ENDUTIH = {"internet_qroo_pct": (92.1, "pág. 10"), "mensajeria_pct": (90.6, "pág. 17"),
           "redes_sociales_pct": (80.4, "pág. 17"), "audio_video_pct": (77.8, "pág. 17")}
LIMITES = {"google_titulo": 30, "google_descripcion": 90, "meta_titulo": 40, "meta_texto": 125}


def cifras() -> dict:
    i = pd.read_parquet(SILVER / "inah")
    q = i[i.es_qroo & (i.anio.astype(int) == 2025)]
    v = q.groupby("region_campana").visitantes.sum()
    nac = q[q.papel_campana == "promovida"].groupby(["region_campana", "tipo_visitante"]).visitantes.sum().unstack()
    n = pd.read_parquet(SILVER / "nacionalidad")
    c = n[(n.lugar_campana == "Cancún") & (n.anio.astype(int) == 2025)]
    tot = c.llegadas_extranjeros.sum()
    pais = c.groupby("pais").llegadas_extranjeros.sum() / tot * 100
    mes = c.groupby("mes").llegadas_extranjeros.sum()
    che = n[(n.lugar_campana == "Chetumal") & (n.anio.astype(int) == 2025)].llegadas_extranjeros.sum()
    return {
        "tulum": int(v["Tulum"]), "ruta": int(v["Ruta arqueológica del sur"]), "bahia": int(v["Bahía Calderitas–Oxtankah"]),
        "veces_ruta": round(v["Tulum"] / v["Ruta arqueológica del sur"], 1),
        "pct_nac_bahia": round(nac.loc["Bahía Calderitas–Oxtankah", "nacional"] / nac.loc["Bahía Calderitas–Oxtankah"].sum() * 100, 1),
        "pct_nac_ruta": round(nac.loc["Ruta arqueológica del sur", "nacional"] / nac.loc["Ruta arqueológica del sur"].sum() * 100, 1),
        "cancun_extranjeros": int(tot), "chetumal_extranjeros": int(che),
        "pct_eeuu": round(pais["Estados Unidos"], 1), "pct_canada": round(pais["Canadá"], 1),
        "pct_mujeres": round(c[c.sexo == "mujer"].llegadas_extranjeros.sum() / tot * 100, 1),
        "mes_pico": int(mes.idxmax()), "mes_bajo": int(mes.idxmin()),
    }


def personas(k: dict, asp: pd.DataFrame) -> list[dict]:
    q = asp[asp.pueblo == "Quintana Roo (3 pueblos)"].set_index("aspecto").riesgo_relativo
    return [
        {"clave": "vuelve", "nombre": "La que vuelve al sur",
         "quien": "Viajera mexicana que ya conoce o tiene cerca el sur de Quintana Roo y busca un viaje con historia, "
                  "comida y calma.",
         "atributos": [
             ("Origen", f"Nacional: {k['pct_nac_bahia']} % de los visitantes de Oxtankah y {k['pct_nac_ruta']} % de los de la Ruta en 2025", "dato", "INAH 2025"),
             ("Estado de origen", "No se publica de qué estado vienen los visitantes nacionales", "hueco", "—"),
             ("Cuándo viaja", "Diciembre a abril (temporada alta del sur); la campaña la invita en mayo–junio y octubre–noviembre, con espacio", "derivado", "Forma del año, Fase 5"),
             ("Qué valora", f"Calma y cultura: una reseña que habla de calma es mala {q['calma']:.2f} veces lo normal; de cultura, {q['cultura']:.2f}", "derivado", "Rest-Mex 2025, texto.py"),
             ("Qué la aleja", f"Ruido ({q['ruido']:.1f}×), suciedad ({q['limpieza']:.1f}×), precio ({q['precio']:.1f}×) y multitudes ({q['multitudes']:.1f}×) en reseñas malas", "derivado", "Rest-Mex 2025"),
             ("Cómo se informa", f"Mensajería ({ENDUTIH['mensajeria_pct'][0]} %) y redes sociales ({ENDUTIH['redes_sociales_pct'][0]} %) en el teléfono", "dato", f"ENDUTIH 2025, {ENDUTIH['mensajeria_pct'][1]}"),
             ("Edad e ingreso", "No hay dato oficial por destino: no se fija", "hueco", "—")]},
        {"clave": "baja", "nombre": "La que baja del norte",
         "quien": "Viajera de Estados Unidos o Canadá que ya está en Cancún o la Riviera Maya y puede bajar al sur en Tren "
                  "Maya o por carretera.",
         "atributos": [
             ("Origen", f"Estados Unidos {k['pct_eeuu']} % y Canadá {k['pct_canada']} % de los {k['cancun_extranjeros']:,} extranjeros que llegaron a Cancún en 2025", "dato", "UPM vía DataTur"),
             ("Sexo", f"Mujeres {k['pct_mujeres']} %", "dato", "UPM vía DataTur"),
             ("Por qué no llega sola al sur", f"El aeropuerto de Chetumal recibió {k['chetumal_extranjeros']} extranjeros en 2025", "dato", "UPM vía DataTur"),
             ("Cuándo está", f"Más llegadas en {['enero','febrero','marzo','abril','mayo','junio','julio','agosto','septiembre','octubre','noviembre','diciembre'][k['mes_pico']-1]}, menos en {['enero','febrero','marzo','abril','mayo','junio','julio','agosto','septiembre','octubre','noviembre','diciembre'][k['mes_bajo']-1]}; Cancún en temporada alta de noviembre a abril", "dato", "UPM; planeador Fase 5"),
             ("Qué la aleja del norte", "Multitudes, precio y sargazo (destinos excluidos por sargazo 2026)", "derivado", "Rest-Mex; REGIONES.md"),
             ("Cómo se informa", "Busca en Google y usa redes desde el destino", "supuesto", "Benchmarks de viajes (D13); sin encuesta propia"),
             ("Idioma", "Inglés (y español)", "derivado", "País de origen")]},
    ]


def marca(k: dict) -> dict:
    return {
        "nombre": "El sur tiene espacio", "lema": "Pirámides, bahía y sabor del sur, con espacio para disfrutarlos.",
        "nombre_en": "The south has room", "lema_en": "Mayan pyramids, a quiet bay and southern food, with room to enjoy them.",
        "personalidad": ["Cercana y tranquila: habla de tú, frases cortas, promete calma.",
                         "Orgullosa de lo maya y de lo mexicano: Kohunlich, Dzibanché, Ichkabal, Oxtankah.",
                         "Con espíritu de aventura: selva, caminos y ruinas para recorrer sin prisa."],
        "propuesta_valor": f"El Caribe mexicano que todavía tiene espacio: zonas mayas con {k['veces_ruta']} veces menos visitantes que Tulum, una bahía tranquila y la comida del sur.",
        "posicionamiento": "Para quien quiere el Caribe mexicano sin multitudes, el sur de Quintana Roo es la opción cultural y tranquila; a diferencia de Tulum o Cancún, aquí hay espacio, y la campaña lo cuida: nunca anuncia un lugar lleno.",
        "visual": "Sistema \"Sur mexicano\" (decisión 07): rosa, amarillo, turquesa y añil en bloques; Bricolage Grotesque y Figtree; greca maya; fotos reales comprobadas en su municipio.",
    }


def mensajes(k: dict) -> list[dict]:
    r = f"INAH 2025: Tulum {k['tulum']:,} visitantes; Ruta {k['ruta']:,} ({k['veces_ruta']} veces menos)"
    return [
        {"persona": "vuelve", "canal": "Google (búsqueda)", "idioma": "es", "lugar": "Ruta arqueológica del sur",
         "titulos": ["Pirámides mayas con espacio", "Kohunlich, Dzibanché, Ichkabal", "El sur tiene espacio"],
         "descripcion": "Recorre la selva y las ruinas del sur sin multitudes. Planea tu viaje con calma.",
         "respaldo": r},
        {"persona": "vuelve", "canal": "Facebook / Instagram", "idioma": "es", "lugar": "Bahía Calderitas–Oxtankah",
         "titulo": "La bahía que te esperaba", "foto": "fotos/lugares/bahia_calderitasoxtankah_3.jpg",
         "texto": "Mariscos frente al mar, una ciudad maya en la orilla y una tarde tranquila. Es nuestro sur.",
         "respaldo": f"Oxtankah recibió {k['bahia']:,} visitantes en 2025 (INAH), {k['pct_nac_bahia']} % mexicanos"},
        {"persona": "vuelve", "canal": "WhatsApp (orgánico)", "idioma": "es", "lugar": "Chetumal",
         "texto": "¿Y si este puente nos vamos al sur? Chetumal, la bahía y las pirámides, con espacio. Mira el plan →",
         "respaldo": f"{ENDUTIH['mensajeria_pct'][0]} % usa mensajería en el teléfono (ENDUTIH 2025, {ENDUTIH['mensajeria_pct'][1]}); sin costo"},
        {"persona": "baja", "canal": "Google (búsqueda)", "idioma": "en", "lugar": "Ruta arqueológica del sur",
         "titulos": ["Mayan pyramids, with room", "The south has room", "Take the Tren Maya south"],
         "descripcion": "Cancún is busy. Southern Quintana Roo has jungle ruins, a calm bay and room for you.",
         "respaldo": f"{r}; Cancún en temporada alta nov–abr (planeador)"},
        {"persona": "baja", "canal": "Facebook / Instagram", "idioma": "en", "lugar": "Ruta arqueológica del sur",
         "titulo": "Your Caribbean trip, one stop south", "foto": "fotos/lugares/ruta_arqueologica_del_sur_2.jpg",
         "texto": "Pyramids in the jungle, few crowds and real southern food. Take the Tren Maya to Chetumal.",
         "respaldo": "Tren Maya con estación en Chetumal (SITUR-Q, descensos por estación); solo se muestra si la Torre "
                     "marca norte lleno y el sur con espacio (regla de la Fase 7)"},
    ]


def revisar_largos(m: list[dict]) -> list[str]:
    errores = []
    for x in m:
        if x["canal"].startswith("Google"):
            errores += [f"título largo: {t}" for t in x["titulos"] if len(t) > LIMITES["google_titulo"]]
            if len(x["descripcion"]) > LIMITES["google_descripcion"]:
                errores.append(f"descripción larga: {x['descripcion']}")
        elif x["canal"].startswith("Facebook"):
            if len(x["titulo"]) > LIMITES["meta_titulo"] or len(x["texto"]) > LIMITES["meta_texto"]:
                errores.append(f"anuncio largo: {x['titulo']}")
    return errores


def medios(k: dict) -> list[dict]:
    pr = pd.read_parquet(GOLD / "presupuesto_plan.parquet").groupby("canal").pesos.sum()
    tot = pr.sum()
    return [
        {"medio": "Facebook / Instagram (pagado)", "pct": round(pr["Facebook"] / tot * 100), "por_que":
         f"Rinde 6.61 visitantes por cada $1,000 (clic de $0.51 USD, WordStream 2025) y {ENDUTIH['redes_sociales_pct'][0]} % usa redes (ENDUTIH 2025). Tope de 70 % para no depender de una plataforma."},
        {"medio": "Google, búsqueda (pagado)", "pct": round(pr["Google"] / tot * 100), "por_que":
         "Llega a quien ya busca viajar (conversión medida de 5.75 % en Travel). Es el canal de la persona que baja del norte."},
        {"medio": "WhatsApp y compartir (orgánico)", "pct": 0, "por_que":
         f"{ENDUTIH['mensajeria_pct'][0]} % usa mensajería: el botón de compartir de la página no cuesta y viaja entre familia y amigos."},
        {"medio": "La página web (destino de todos los anuncios)", "pct": 0, "por_que":
         "Planeador por lugar y mes en 9 idiomas, sin internet externo; recomienda otro lugar del sur si el elegido está lleno."},
    ]


def calendario() -> list[dict]:
    plan = pd.read_parquet(GOLD / "presupuesto_plan.parquet")
    cal = pd.read_parquet(GOLD / "pronostico_calendario.parquet")
    alta_cancun = set(cal[(cal.lugar == "Cancún") & (cal.nivel == "alta")].periodo)
    filas = []
    for p, g in plan.groupby("periodo"):
        lug = g.groupby("lugar").pesos.sum()
        filas.append({"mes": f"{pd.Timestamp(p):%Y-%m}", "pesos": round(float(lug.sum())),
                      "lugares": [l for l, v in lug.items() if v > 0.5],
                      "personas": ([] if lug.sum() < 0.5 else ["vuelve"] + (["baja"] if p in alta_cancun else []))})
    return filas


def kpis(k: dict) -> list[dict]:
    pz = pd.read_parquet(GOLD / "presupuesto_sensibilidad.parquet").query("caso == 'Base'").iloc[0]
    return [
        {"kpi": "Alcance", "formula": "Personas únicas que vieron un anuncio", "meta": "Se fija con el primer mes real (no hay dato previo)", "fuente": "Plataformas", "frecuencia": "Semanal"},
        {"kpi": "Interacción (CTR)", "formula": "Clics ÷ impresiones", "meta": "Google ≥ 8.73 %; Facebook ≥ 2.76 % (promedios de Travel)", "fuente": "Plataformas; WordStream 2025", "frecuencia": "Semanal"},
        {"kpi": "Conversión", "formula": "Planes de viaje o contactos ÷ clics", "meta": "≥ 5.75 % (Google Travel)", "fuente": "Página web", "frecuencia": "Semanal"},
        {"kpi": "Costo por visitante", "formula": "Pesos gastados ÷ conversiones", "meta": f"≤ ${pz.pesos_por_visitante:,.0f} (modelo de la Fase 6)", "fuente": "Plataformas + página", "frecuencia": "Mensual"},
        {"kpi": "Afluencia", "formula": "Visitantes reales − escenario probable del pronóstico", "meta": f"+{pz.visitantes_esperados:,.0f} en 9 meses, sin pasar la capacidad probada", "fuente": "INAH, SITUR-Q (Belice)", "frecuencia": "Mensual"},
        {"kpi": "Presupuesto", "formula": "Pesos gastados ÷ pesos planeados", "meta": "100 % (lo pausado se gasta después)", "fuente": "Torre en vivo", "frecuencia": "Semanal"},
        {"kpi": "Económico", "formula": "Ocupación hotelera de Chetumal", "meta": "Hueco: SITUR-Q no publica ocupación del sur desde 2025", "fuente": "—", "frecuencia": "—"},
        {"kpi": "Social", "formula": "Visitantes por cada 1,000 habitantes (Radar)", "meta": "El lugar sigue \"tranquilo\" o \"concurrido\", nunca \"saturado\"", "fuente": "Radar, Fase 4", "frecuencia": "Mensual"},
        {"kpi": "Ambiental", "formula": "Semanas con anuncio en temporada alta o con mal clima", "meta": "0 (la Torre las pausa)", "fuente": "Torre en vivo, Fase 7", "frecuencia": "Semanal"},
        {"kpi": "Capacidad", "formula": "(Esperados + campaña) ÷ capacidad probada", "meta": "≤ 40 % en los meses con anuncio (Pareto, Fase 6)", "fuente": "Pronóstico + presupuesto", "frecuencia": "Mensual"},
    ]


def construir() -> dict:
    k = cifras()
    asp = pd.read_parquet(GOLD / "campana_aspectos.parquet")
    pal = pd.read_parquet(GOLD / "campana_palabras.parquet")
    reg = pd.read_parquet(GOLD / "campana_reglas.parquet")
    m = mensajes(k)
    errores = revisar_largos(m)
    if errores:
        raise ValueError(f"Anuncios fuera de los límites de la plataforma: {errores}")
    q = asp[asp.pueblo == "Quintana Roo (3 pueblos)"].sort_values("riesgo_relativo", ascending=False)
    c = {"cifras": k, "personas": personas(k, asp), "marca": marca(k), "mensajes": m, "medios": medios(k),
         "calendario": calendario(), "kpis": kpis(k),
         "texto": {"aspectos": q[["aspecto", "pct_menciona", "riesgo_relativo"]].round(2).to_dict("records"),
                   "palabras_cinco": pal[pal.lado.str.contains("5")].palabra.head(12).tolist(),
                   "palabras_malas": pal[pal.lado.str.contains("1–2")].palabra.head(12).tolist(),
                   "reglas_malas": reg[reg.entonces.str.contains("mala")].head(5).round(3).to_dict("records")},
         "endutih": {k2: v[0] for k2, v in ENDUTIH.items()}}
    SALIDA.write_text(json.dumps(c, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
    print(f"Campaña \"{c['marca']['nombre']}\": {len(c['personas'])} personas, {len(m)} mensajes (largos OK), "
          f"{len(c['calendario'])} meses, {len(c['kpis'])} KPI → {SALIDA.relative_to(RAIZ)}")
    for p in c["personas"]:
        print(f"  {p['nombre']}: " + "; ".join(f"{a} [{t}]" for a, _, t, _ in p["atributos"]))
    return c


### Se arma la campaña
`texto.construir()` mide los temas, las reglas de asociación y las palabras de las reseñas; `marca.construir()` arma las
personas, la marca, los mensajes, los medios, el calendario y los indicadores con las cifras de los otros notebooks, y lo
guarda en `datos/gold/campana.json`.

In [5]:
from torre.campana import texto, marca
texto.construir()
marca.construir();

Reseñas: 85,987 · malas (1–2): 4.2% · de 5: 69.9%
   aspecto  pct_menciona  pct_malas  riesgo_relativo
     ruido          2.37      11.53             2.76
  limpieza         11.07       7.92             1.89
    precio         25.58       6.96             1.66
multitudes         17.95       6.35             1.52
  servicio         44.24       4.95             1.18
    comida         40.35       3.40             0.81
naturaleza         13.92       3.06             0.73
   cultura         18.28       1.99             0.48
     calma         12.29       1.72             0.41
pueblo      Bacalar  Isla Mujeres  Quintana Roo (3 pueblos)  Tulum
aspecto                                                           
calma          0.18          0.43                      0.41   0.47
comida         0.68          0.70                      0.81   0.97
cultura        0.19          0.89                      0.48   0.50
limpieza       1.97          1.75                      1.89   1.99
multitudes     1.8

Campaña "El sur tiene espacio": 2 personas, 5 mensajes (largos OK), 9 meses, 10 KPI → datos\gold\campana.json
  La que vuelve al sur: Origen [dato]; Estado de origen [hueco]; Cuándo viaja [derivado]; Qué valora [derivado]; Qué la aleja [derivado]; Cómo se informa [dato]; Edad e ingreso [hueco]
  La que baja del norte: Origen [dato]; Sexo [dato]; Por qué no llega sola al sur [dato]; Cuándo está [dato]; Qué la aleja del norte [derivado]; Cómo se informa [supuesto]; Idioma [derivado]


## La campaña · 07 · La campaña: "El sur tiene espacio"
**Torre del Caribe** · Fase 8 (Mercadotecnia digital) · Autor: **Brandon Uriel García Sánchez** · UNRC, LCDN 5° semestre 2026-2

| | |
|---|---|
| **Qué hace** | Convierte los datos en una campaña: qué molesta y qué enamora en las reseñas, a quién se le habla, con qué palabras, en qué canal, cuándo y cómo se mide. |
| **Decisiones del equipo** | Dos personas (la que vuelve al sur; la que baja del norte) · nombre "El sur tiene espacio" · tono que combina calma, orgullo cultural y aventura (*Las decisiones, fase por fase* (PDF)). |
| **Código** | `torre.campana.texto` (minería) y `` (campaña). |
| **Ecuaciones** | *Guía técnica* (PDF) |

In [6]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import pandas as pd
from torre.campana import texto as tx, marca as mk
pd.set_option("display.max_colwidth", 120)

### 1. Temas de las reseñas y su riesgo
Léxico a la vista (`tx.ASPECTOS`). Riesgo relativo $RR_a=\dfrac{P(\text{mala}\mid a)}{P(\text{mala})}$: más de 1, el tema
aleja; menos de 1, protege.

In [7]:
r = tx.marcar_aspectos(tx.leer())
print(f"{len(r):,} reseñas · malas {r.mala.mean():.2%} · de 5 estrellas {r.cinco.mean():.2%}")
a = tx.aspectos(r)
a[a.pueblo == "Quintana Roo (3 pueblos)"].sort_values("riesgo_relativo", ascending=False).round(2)

85,987 reseñas · malas 4.18% · de 5 estrellas 69.87%


,pueblo,aspecto,resenas,pct_menciona,pct_malas,pct_malas_pueblo,riesgo_relativo
30,Quintana Roo (3 pueblos),ruido,2038,2.37,11.53,4.18,2.76
29,Quintana Roo (3 pueblos),limpieza,9520,11.07,7.92,4.18,1.89
28,Quintana Roo (3 pueblos),precio,21999,25.58,6.96,4.18,1.66
27,Quintana Roo (3 pueblos),multitudes,15434,17.95,6.35,4.18,1.52
35,Quintana Roo (3 pueblos),servicio,38041,44.24,4.95,4.18,1.18
34,Quintana Roo (3 pueblos),comida,34696,40.35,3.40,4.18,0.81
32,Quintana Roo (3 pueblos),naturaleza,11971,13.92,3.06,4.18,0.73
33,Quintana Roo (3 pueblos),cultura,15721,18.28,1.99,4.18,0.48
31,Quintana Roo (3 pueblos),calma,10572,12.29,1.72,4.18,0.41


### 2. Reglas de asociación (Apriori)
Transacciones = temas mencionados + la calificación. soporte $=\#\{A\cup B\}/N$, confianza $=sop(A\cup B)/sop(A)$,
lift $=conf/sop(B)$.

In [8]:
reg = tx.reglas_asociacion(r)
display(reg[reg.entonces.str.contains("mala")].head(8).round(4))
reg[reg.entonces == "reseña de 5"].head(6).round(4)

,si,entonces,soporte,confianza,lift
446,ruido + servicio,reseña mala (1–2),0.0017,0.1179,2.8194
40,ruido,reseña mala (1–2),0.0027,0.1153,2.7565
138,limpieza + multitudes,reseña mala (1–2),0.0027,0.1084,2.5924
256,limpieza + precio,reseña mala (1–2),0.0039,0.1062,2.5395
339,precio + servicio,reseña mala (1–2),0.0094,0.0851,2.0337
98,multitudes + precio,reseña mala (1–2),0.0047,0.0837,2.0007
221,multitudes + servicio,reseña mala (1–2),0.0059,0.0824,1.9708
35,limpieza,reseña mala (1–2),0.0088,0.0792,1.8933


,si,entonces,soporte,confianza,lift
514,comida + servicio,reseña de 5,0.2010,0.7813,1.1182
462,calma + naturaleza,reseña de 5,0.0172,0.7663,1.0967
52,comida,reseña de 5,0.3053,0.7567,1.0829
480,calma + comida,reseña de 5,0.0361,0.7553,1.0809
484,calma + servicio,reseña de 5,0.0456,0.7504,1.0740
56,servicio,reseña de 5,0.3280,0.7414,1.0610


### 3. El lenguaje del turista (log-odds con prior de Dirichlet informativo)

In [9]:
p = tx.palabras_que_distinguen(r)
p.groupby("lado").palabra.apply(lambda x: ", ".join(x))

lado
reseñas de 1–2 estrellas    habitación, dijeron, peor, horrible, dinero, terrible, caro, hotel, dijo, decepcionante, nadie, decepción, pedimos, ...
reseñas de 5 estrellas      excelente, increíble, playa, tulum, comida, isla, ruinas, hermoso, deliciosa, amable, encantó, ambiente, gracias, du...
Name: palabra, dtype: object

### 4. Las dos viajeras ideales
Cada rasgo dice si es **dato**, **derivado**, **supuesto** o **hueco**.

In [10]:
c = json.loads((RAIZ / "datos" / "gold" / "campana.json").read_text(encoding="utf-8"))
for persona in c["personas"]:
    print("\n" + persona["nombre"].upper(), "—", persona["quien"])
    display(pd.DataFrame(persona["atributos"], columns=["rasgo", "valor", "tipo", "fuente"]))


LA QUE VUELVE AL SUR — Viajera mexicana que ya conoce o tiene cerca el sur de Quintana Roo y busca un viaje con historia, comida y calma.


,rasgo,valor,tipo,fuente
0,Origen,Nacional: 94.8 % de los visitantes de Oxtankah y 63.3 % de los de la Ruta en 2025,dato,INAH 2025
1,Estado de origen,No se publica de qué estado vienen los visitantes nacionales,hueco,—
2,Cuándo viaja,"Diciembre a abril (temporada alta del sur); la campaña la invita en mayo–junio y octubre–noviembre, con espacio",derivado,"Forma del año, Fase 5"
3,Qué valora,"Calma y cultura: una reseña que habla de calma es mala 0.41 veces lo normal; de cultura, 0.48",derivado,"Rest-Mex 2025, texto.py"
4,Qué la aleja,"Ruido (2.8×), suciedad (1.9×), precio (1.7×) y multitudes (1.5×) en reseñas malas",derivado,Rest-Mex 2025
5,Cómo se informa,Mensajería (90.6 %) y redes sociales (80.4 %) en el teléfono,dato,"ENDUTIH 2025, pág. 17"
6,Edad e ingreso,No hay dato oficial por destino: no se fija,hueco,—



LA QUE BAJA DEL NORTE — Viajera de Estados Unidos o Canadá que ya está en Cancún o la Riviera Maya y puede bajar al sur en Tren Maya o por carretera.


,rasgo,valor,tipo,fuente
0,Origen,"Estados Unidos 56.3 % y Canadá 17.1 % de los 9,408,423 extranjeros que llegaron a Cancún en 2025",dato,UPM vía DataTur
1,Sexo,Mujeres 53.4 %,dato,UPM vía DataTur
2,Por qué no llega sola al sur,El aeropuerto de Chetumal recibió 250 extranjeros en 2025,dato,UPM vía DataTur
3,Cuándo está,"Más llegadas en marzo, menos en septiembre; Cancún en temporada alta de noviembre a abril",dato,UPM; planeador Fase 5
4,Qué la aleja del norte,"Multitudes, precio y sargazo (destinos excluidos por sargazo 2026)",derivado,Rest-Mex; REGIONES.md
5,Cómo se informa,Busca en Google y usa redes desde el destino,supuesto,Benchmarks de viajes (D13); sin encuesta propia
6,Idioma,Inglés (y español),derivado,País de origen


### 5. Marca, mensajes, medios, calendario y KPI

In [11]:
print(json.dumps(c["marca"], ensure_ascii=False, indent=1))
print("Revisión de largos de anuncios:", mk.revisar_largos(c["mensajes"]) or "todo dentro de los límites")
display(pd.DataFrame(c["mensajes"])[["persona", "canal", "idioma", "respaldo"]])
display(pd.DataFrame(c["medios"]))
display(pd.DataFrame(c["calendario"]))
pd.DataFrame(c["kpis"])

{
 "nombre": "El sur tiene espacio",
 "lema": "Pirámides, bahía y sabor del sur, con espacio para disfrutarlos.",
 "nombre_en": "The south has room",
 "lema_en": "Mayan pyramids, a quiet bay and southern food, with room to enjoy them.",
 "personalidad": [
  "Cercana y tranquila: habla de tú, frases cortas, promete calma.",
  "Orgullosa de lo maya y de lo mexicano: Kohunlich, Dzibanché, Ichkabal, Oxtankah.",
  "Con espíritu de aventura: selva, caminos y ruinas para recorrer sin prisa."
 ],
 "propuesta_valor": "El Caribe mexicano que todavía tiene espacio: zonas mayas con 15.5 veces menos visitantes que Tulum, una bahía tranquila y la comida del sur.",
 "posicionamiento": "Para quien quiere el Caribe mexicano sin multitudes, el sur de Quintana Roo es la opción cultural y tranquila; a diferencia de Tulum o Cancún, aquí hay espacio, y la campaña lo cuida: nunca anuncia un lugar lleno.",
 "visual": "Sistema \"Sur mexicano\" (decisión 07): rosa, amarillo, turquesa y añil en bloques; Bricolag

,persona,canal,idioma,respaldo
0,vuelve,Google (búsqueda),es,"INAH 2025: Tulum 1,031,443 visitantes; Ruta 66,628 (15.5 veces menos)"
1,vuelve,Facebook / Instagram,es,"Oxtankah recibió 11,017 visitantes en 2025 (INAH), 94.8 % mexicanos"
2,vuelve,WhatsApp (orgánico),es,"90.6 % usa mensajería en el teléfono (ENDUTIH 2025, pág. 17); sin costo"
3,baja,Google (búsqueda),en,"INAH 2025: Tulum 1,031,443 visitantes; Ruta 66,628 (15.5 veces menos); Cancún en temporada alta nov–abr (planeador)"
4,baja,Facebook / Instagram,en,"Tren Maya con estación en Chetumal (SITUR-Q, descensos por estación); solo se muestra si la Torre marca norte lleno ..."


,medio,pct,por_que
0,Facebook / Instagram (pagado),70,"Rinde 6.61 visitantes por cada $1,000 (clic de $0.51 USD, WordStream 2025) y 80.4 % usa redes (ENDUTIH 2025). Tope d..."
1,"Google, búsqueda (pagado)",30,Llega a quien ya busca viajar (conversión medida de 5.75 % en Travel). Es el canal de la persona que baja del norte.
2,WhatsApp y compartir (orgánico),0,90.6 % usa mensajería: el botón de compartir de la página no cuesta y viaja entre familia y amigos.
3,La página web (destino de todos los anuncios),0,"Planeador por lugar y mes en 9 idiomas, sin internet externo; recomienda otro lugar del sur si el elegido está lleno."


,mes,pesos,lugares,personas
0,2026-10,32586,"[Bahía Calderitas–Oxtankah, Chetumal, Ruta arqueológica del sur]",[vuelve]
1,2026-11,28012,"[Bahía Calderitas–Oxtankah, Chetumal, Ruta arqueológica del sur]","[vuelve, baja]"
2,2026-12,0,[],[]
3,2027-01,5650,[Chetumal],"[vuelve, baja]"
4,2027-02,25339,"[Bahía Calderitas–Oxtankah, Chetumal, Ruta arqueológica del sur]","[vuelve, baja]"
5,2027-03,12073,"[Bahía Calderitas–Oxtankah, Chetumal]","[vuelve, baja]"
6,2027-04,14496,"[Chetumal, Ruta arqueológica del sur]","[vuelve, baja]"
7,2027-05,33359,"[Bahía Calderitas–Oxtankah, Chetumal, Ruta arqueológica del sur]",[vuelve]
8,2027-06,35985,"[Bahía Calderitas–Oxtankah, Chetumal, Ruta arqueológica del sur]",[vuelve]


,kpi,formula,meta,fuente,frecuencia
0,Alcance,Personas únicas que vieron un anuncio,Se fija con el primer mes real (no hay dato previo),Plataformas,Semanal
1,Interacción (CTR),Clics ÷ impresiones,Google ≥ 8.73 %; Facebook ≥ 2.76 % (promedios de Travel),Plataformas; WordStream 2025,Semanal
2,Conversión,Planes de viaje o contactos ÷ clics,≥ 5.75 % (Google Travel),Página web,Semanal
3,Costo por visitante,Pesos gastados ÷ conversiones,≤ $196 (modelo de la Fase 6),Plataformas + página,Mensual
4,Afluencia,Visitantes reales − escenario probable del pronóstico,"+957 en 9 meses, sin pasar la capacidad probada","INAH, SITUR-Q (Belice)",Mensual
5,Presupuesto,Pesos gastados ÷ pesos planeados,100 % (lo pausado se gasta después),Torre en vivo,Semanal
6,Económico,Ocupación hotelera de Chetumal,Hueco: SITUR-Q no publica ocupación del sur desde 2025,—,—
7,Social,"Visitantes por cada 1,000 habitantes (Radar)","El lugar sigue ""tranquilo"" o ""concurrido"", nunca ""saturado""","Radar, Fase 4",Mensual
8,Ambiental,Semanas con anuncio en temporada alta o con mal clima,0 (la Torre las pausa),"Torre en vivo, Fase 7",Semanal
9,Capacidad,(Esperados + campaña) ÷ capacidad probada,"≤ 40 % en los meses con anuncio (Pareto, Fase 6)",Pronóstico + presupuesto,Mensual


### 6. Qué se concluye
1. **Lo que hunde una reseña es lo que sobra en los destinos llenos:** ruido (2.76×), suciedad (1.89×), precio (1.66×) y
   multitudes (1.52×). Lo que la protege es lo que el sur ofrece: calma (0.41×) y cultura (0.48×).
2. **Dos viajeras, dos caminos:** la nacional ya es el 95 % de la Bahía; la extranjera está en Cancún (9.4 millones en
   2025) y casi no llega al sur por avión (250 en Chetumal). Por eso hay un mensaje para volver y otro para bajar.
3. **Ningún anuncio promete lo que no está medido:** sin precios ni horas de viaje; "con espacio" se apoya en 15.5 veces
   menos visitantes que Tulum.
4. **Limitaciones:** Rest-Mex no cubre los cinco lugares; muchas reseñas son traducciones; no hay dato del estado de
   origen del visitante nacional ni de edad o ingreso; los canales y conversiones son promedios de EE. UU.